In [12]:
import pandas as pd
import numpy as np
import xgboost as xgb
import pickle
from sklearn.preprocessing import OrdinalEncoder

In [13]:
df = pd.read_csv("src/data/inference/raw/merged_inference_data.csv")

In [14]:
user_ids = df['user_id']

In [15]:
df["mix__sd_od_history_to_kyc_ratio"] = df["od__history_length_days"] / df["sd__age_since_first_kyc"]

In [16]:
ordinal_features = [
    "cb__worst_score",
    "sd__membership_product"
]

ordinal_encoder = OrdinalEncoder(
    categories=[
        # example – replace with your actual order
        ["A", "B", "C", "D", "E", "F", "G", "M", "N", "O", "P", np.nan],           # cb__worst_score
        ["PERSONAL_FLEX", "PERSONAL_STANDARD", "PERSONAL_SMART"
         , "PERSONAL_YOU", "PERSONAL_METAL", "BUSINESS_STANDARD"
         , "BUSINESS_SMART", "BUSINESS_YOU", "BUSINESS_METAL", np.nan]        # sd__membership_product
    ]
)

df[ordinal_features] = ordinal_encoder.fit_transform(
    df[ordinal_features]
)

In [17]:
with open("xgb_clf_ccf.pkl", "rb") as f:
    best_clf_model = pickle.load(f)

with open("xgb_reg_ccf.pkl", "rb") as f:
    best_reg_model = pickle.load(f)

In [20]:
clf_features = best_clf_model.get_booster().feature_names
reg_features = best_reg_model.get_booster().feature_names

X_clf = df[clf_features]
X_reg = df[reg_features]

In [8]:
threshold = 0.223114

def predict_ccf(X_clf, X_reg, best_clf_model, best_reg_model, threshold=0.223114, upper_clipping_limit=1.0, bottom_clipping_limit=0.0):
    """
    Applies the Hurdle Model logic to predict CCF for new data.
    """
    # 1. Classification (The Hurdle)
    clf_probabilities = best_clf_model.predict_proba(X_clf)[:, 1]
    predicted_ccf = np.zeros(X_clf.shape[0])
    non_zero_indices = clf_probabilities >= threshold
    
    if np.any(non_zero_indices):
        # 2. Regression (The Value)
        X_reg_input = X_reg[non_zero_indices]
        reg_predictions = best_reg_model.predict(X_reg_input)
        
        # 3. Clipping (The Constraint)
        final_values = np.clip(reg_predictions, a_min=bottom_clipping_limit, a_max=upper_clipping_limit)
        
        # 4. Final Output Assembly
        predicted_ccf[non_zero_indices] = final_values
        
    return predicted_ccf

In [48]:
THRESHOLD = 0.223114  # example – use your calibrated value

df["ccf"] = predict_ccf(
    X_clf=X_clf,
    X_reg=X_reg,
    best_clf_model=best_clf_model,
    best_reg_model=best_reg_model,
    threshold=THRESHOLD
)

In [27]:
y_pred_clf = best_clf_model.predict_proba(X_clf)[:,1]
y_pred_reg = best_reg_model.predict(X_reg)

df["ccf"] = y_pred_clf*np.clip(a=y_pred_reg, a_min=0.0, a_max=1)

In [28]:
df

,user_id,reference_date,ob__limit_ref,ob__balance_ref,ob__open_limit_ref,ob__avg_util_ref,mv__is_lisbon_v1,mv__is_lisbon_v2,mv__is_lisbon_v3,mv__is_lisbon_v4,...,ab__inflow_spike_count_6m,ab__days_down_6m,ab__near_zero_days_6m,dn__recent_action,dn__max_action_level,cb__most_recent_score,cb__worst_score,ep__cc_tbil_balance,mix__sd_od_history_to_kyc_ratio,ccf
0,15290ec6-09c3-4e11-b28f-40e0ca4886f7,2026-02-28,3750.0,0.00,1.00000,0.00000,1,0,0,0,...,7,45,0,NaN,NaN,F,5.0,0.00,1.000373,0.245691
1,1587a2dc-c0b0-4120-8002-0c1cde9211fa,2026-02-28,8000.0,0.00,1.00000,0.00000,1,0,0,0,...,5,78,0,NaN,NaN,C,7.0,1473.87,0.813255,0.277708
2,15ad0b22-759c-41e2-8036-01012299aee4,2026-02-28,500.0,476.69,0.04662,0.95338,0,0,0,0,...,8,46,2,NaN,NaN,M,7.0,4407.33,0.225092,0.478020
3,16e500cb-26e5-409b-b7eb-a963e1235bc7,2026-02-28,250.0,0.00,1.00000,0.00000,1,0,0,0,...,0,0,181,NaN,NaN,D,6.0,0.00,0.852490,0.099179
4,17061379-646d-4e63-a919-62dfc64d8930,2026-02-28,500.0,508.22,0.00000,1.01644,1,0,0,0,...,0,7,0,NaN,NaN,E,4.0,0.00,0.933815,0.700028
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
168491,1301ebda-d9df-4cf0-8958-494d292d05c9,2026-02-28,10000.0,0.00,1.00000,0.00000,1,0,0,0,...,10,142,0,NaN,NaN,C,7.0,0.00,0.498038,0.131307
168492,130f31bd-a952-4e9b-9724-3aec0049c718,2026-02-28,500.0,0.00,1.00000,0.00000,1,0,0,0,...,6,86,34,NaN,NaN,F,5.0,0.00,0.954778,0.242696
168493,1345d8ef-864c-489b-94bf-53a9c637f5aa,2026-02-28,1000.0,0.00,1.00000,0.00000,0,0,0,0,...,15,133,2,NaN,NaN,G,7.0,929.10,0.627151,0.287267
168494,135b6514-8cd2-492a-9bb2-2a9528f96abc,2026-02-28,250.0,66.13,0.73548,0.26452,1,0,0,0,...,1,19,5,NaN,NaN,D,6.0,0.00,0.668569,0.369606


In [29]:
final_df = df[['user_id', 'reference_date', 'ccf']]

In [30]:
final_df.to_csv("ccf__output_28.02.2026.csv", index=False)

In [31]:
df['ccf'].mean()

np.float32(0.30054197)

In [33]:
final_df

,user_id,reference_date,ccf
0,15290ec6-09c3-4e11-b28f-40e0ca4886f7,2026-02-28,0.245691
1,1587a2dc-c0b0-4120-8002-0c1cde9211fa,2026-02-28,0.277708
2,15ad0b22-759c-41e2-8036-01012299aee4,2026-02-28,0.478020
3,16e500cb-26e5-409b-b7eb-a963e1235bc7,2026-02-28,0.099179
4,17061379-646d-4e63-a919-62dfc64d8930,2026-02-28,0.700028
...,...,...,...
168491,1301ebda-d9df-4cf0-8958-494d292d05c9,2026-02-28,0.131307
168492,130f31bd-a952-4e9b-9724-3aec0049c718,2026-02-28,0.242696
168493,1345d8ef-864c-489b-94bf-53a9c637f5aa,2026-02-28,0.287267
168494,135b6514-8cd2-492a-9bb2-2a9528f96abc,2026-02-28,0.369606
